In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


Loading Datasets (mnist, cifar10_gray, madelon)

In [2]:
class DatasetLoader:
    """
    Handles loading, preprocessing, and splitting of datasets.
    Designed for feature selection experiments.
    """

    def __init__(self, dataset_name: str):
        self.dataset_name = dataset_name
        self.X = None
        self.y = None
        self.feature_names = None

    def load(self):
        print(f"\n📦 Loading dataset: {self.dataset_name} ...")

        if self.dataset_name == "mnist":
            data = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
            self.X = data.data.astype(np.float32)
            self.y = data.target.astype(int)
            self.feature_names = [f"pixel_{i}" for i in range(self.X.shape[1])]

        elif self.dataset_name == "cifar10_gray":
            from keras.datasets import cifar10
            import cv2
            (X_train, y_train), (X_test, y_test) = cifar10.load_data()
            # Convert to grayscale and flatten → 1024 features
            X_all = np.concatenate([X_train, X_test], axis=0)
            y_all = np.concatenate([y_train.flatten(), y_test.flatten()], axis=0)
            X_gray = np.array([
                cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).flatten()
                for img in X_all
            ], dtype=np.float32)
            self.X = X_gray
            self.y = y_all
            self.feature_names = [f"pixel_{i}" for i in range(self.X.shape[1])]

        elif self.dataset_name == "madelon":
            data = fetch_openml('madelon', version=1, as_frame=False, parser='auto')
            self.X = data.data.astype(np.float32)
            self.y = (data.target.astype(int) > 0).astype(int)  # binary
            self.feature_names = [f"feat_{i}" for i in range(self.X.shape[1])]

        else:
            raise ValueError(f"Unknown dataset: {self.dataset_name}")

        print(f"   Shape     : {self.X.shape}")
        print(f"   Classes   : {np.unique(self.y)}")
        print(f"   Features  : {self.X.shape[1]}")
        return self

    def preprocess(self, sample_size=None):
        """Scale features to [0,1]. Optionally subsample for speed."""
        if sample_size and sample_size < len(self.X):
            idx = np.random.choice(len(self.X), sample_size, replace=False)
            self.X = self.X[idx]
            self.y = self.y[idx]
            print(f"   Subsampled to {sample_size} samples")

        scaler = MinMaxScaler()
        self.X = scaler.fit_transform(self.X)
        print(f"   ✅ Preprocessing done. Feature range: [{self.X.min():.2f}, {self.X.max():.2f}]")
        return self

    def split(self, test_size=0.2, random_state=42):
        X_train, X_test, y_train, y_test = train_test_split(
            self.X, self.y, test_size=test_size, random_state=random_state, stratify=self.y
        )
        print(f"   Train: {X_train.shape}, Test: {X_test.shape}")
        return X_train, X_test, y_train, y_test

Naive Bayes model for evaluation

In [3]:
class NaiveBayesEvaluator:
    """
    Wraps GaussianNaiveBayes for use as the ML evaluator
    in the feature selection pipeline.
    This will be called by the Fitness Function
    """

    def __init__(self):
        self.model = GaussianNB()

    def evaluate(self, X_train, X_test, y_train, y_test, feature_mask=None):
        """
        Evaluate accuracy using a binary feature mask.
        feature_mask: array of 0s and 1s (e.g., [1,0,1,1,...])
                      If None, uses ALL features (baseline).
        Returns: accuracy (float), time_taken (float)
        """
        if feature_mask is not None:
            selected = np.where(np.array(feature_mask) == 1)[0]
            if len(selected) == 0:
                return 0.0, 0.0
            X_tr = X_train[:, selected]
            X_te = X_test[:, selected]
        else:
            X_tr, X_te = X_train, X_test

        start = time.time()
        self.model.fit(X_tr, y_train)
        preds = self.model.predict(X_te)
        elapsed = time.time() - start

        acc = accuracy_score(y_test, preds)
        return acc, elapsed

Loading and Pre-processing Datasets

In [4]:
# ─────────────────────────────────────────────
# 1. DATASET LOADER
# ─────────────────────────────────────────────
class DatasetLoader:
    def __init__(self, dataset_name: str):
        self.dataset_name = dataset_name
        self.X = None
        self.y = None

    def load(self):
        print(f"\n📦 Loading dataset: {self.dataset_name} ...")

        if self.dataset_name == "mnist":
          from keras.datasets import mnist
          (X_train, y_train), (X_test, y_test) = mnist.load_data()
          X_all = np.concatenate([X_train, X_test], axis=0)
          y_all = np.concatenate([y_train, y_test], axis=0)
          # Flatten 28x28 images into 784 features
          self.X = X_all.reshape(X_all.shape[0], -1).astype(np.float32)
          self.y = y_all.astype(int)

        elif self.dataset_name == "cifar10_gray":
            from keras.datasets import cifar10
            import cv2
            (X_train, y_train), (X_test, y_test) = cifar10.load_data()
            X_all = np.concatenate([X_train, X_test], axis=0)
            y_all = np.concatenate([y_train.flatten(), y_test.flatten()], axis=0)
            self.X = np.array([
                cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).flatten()
                for img in X_all
            ], dtype=np.float32)
            self.y = y_all

        elif self.dataset_name == "madelon":
            from sklearn.datasets import make_classification
            X, y = make_classification(
                n_samples=2600,
                n_features=500,
                n_informative=20,
                n_redundant=480,
                n_classes=2,
                random_state=42
            )
            self.X = X.astype(np.float32)
            self.y = y.astype(int)

        else:
            raise ValueError(f"Unknown dataset: {self.dataset_name}")

        print(f"   Shape     : {self.X.shape}")
        print(f"   Classes   : {np.unique(self.y)}")
        print(f"   Features  : {self.X.shape[1]}")
        return self

    def preprocess(self, sample_size=None):
        if sample_size and sample_size < len(self.X):
            np.random.seed(42)
            idx = np.random.choice(len(self.X), sample_size, replace=False)
            self.X = self.X[idx]
            self.y = self.y[idx]
            print(f"   Subsampled to {sample_size} samples")

        scaler = MinMaxScaler()
        self.X = scaler.fit_transform(self.X)
        print(f"   ✅ Preprocessing done. Feature range: [{self.X.min():.2f}, {self.X.max():.2f}]")
        return self

    def split(self, test_size=0.2, random_state=42):
        X_train, X_test, y_train, y_test = train_test_split(
            self.X, self.y,
            test_size=test_size,
            random_state=random_state,
            stratify=self.y
        )
        print(f"   Train: {X_train.shape}, Test: {X_test.shape}")
        return X_train, X_test, y_train, y_test



Naive Bayes Evaluator

In [5]:
# ─────────────────────────────────────────────
# 2. NAIVE BAYES EVALUATOR
# ─────────────────────────────────────────────
class NaiveBayesEvaluator:
    def __init__(self):
        self.model = GaussianNB()

    def evaluate(self, X_train, X_test, y_train, y_test, feature_mask=None):
        if feature_mask is not None:
            selected = np.where(np.array(feature_mask) == 1)[0]
            if len(selected) == 0:
                return 0.0, 0.0
            X_tr = X_train[:, selected]
            X_te = X_test[:, selected]
        else:
            X_tr, X_te = X_train, X_test

        start = time.time()
        self.model.fit(X_tr, y_train)
        preds = self.model.predict(X_te)
        elapsed = time.time() - start

        acc = accuracy_score(y_test, preds)
        return acc, elapsed


# ─────────────────────────────────────────────
# 3. BASELINE RUNNER
# ─────────────────────────────────────────────
def run_baseline(dataset_name, sample_size=5000):
    print(f"\n{'='*50}")
    print(f"  BASELINE: {dataset_name.upper()}")
    print(f"{'='*50}")

    loader = DatasetLoader(dataset_name)
    loader.load().preprocess(sample_size=sample_size)
    X_train, X_test, y_train, y_test = loader.split()

    evaluator = NaiveBayesEvaluator()
    acc, t = evaluator.evaluate(X_train, X_test, y_train, y_test, feature_mask=None)

    # Sanity check — print both train and test accuracy for madelon
    if dataset_name == "madelon":
        train_acc = accuracy_score(y_train, evaluator.model.predict(X_train))
        print(f"\n🔍 Sanity Check:")
        print(f"   Train Accuracy : {train_acc * 100:.2f}%")
        print(f"   Test  Accuracy : {acc * 100:.2f}%")

    print(f"\n📊 Results (ALL {X_train.shape[1]} features):")
    print(f"   Accuracy : {acc * 100:.2f}%")
    print(f"   Time     : {t:.4f}s")

    return {
        "dataset": dataset_name,
        "n_features": X_train.shape[1],
        "accuracy_full": acc,
        "time_full": t,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test
    }


# ─────────────────────────────────────────────
# 4. RUN ALL 3 DATASETS
# ─────────────────────────────────────────────
results = {}
results["mnist"]        = run_baseline("mnist",        sample_size=5000)
results["madelon"]      = run_baseline("madelon",      sample_size=None)
results["cifar10_gray"] = run_baseline("cifar10_gray", sample_size=5000)



  BASELINE: MNIST

📦 Loading dataset: mnist ...
   Shape     : (70000, 784)
   Classes   : [0 1 2 3 4 5 6 7 8 9]
   Features  : 784
   Subsampled to 5000 samples
   ✅ Preprocessing done. Feature range: [0.00, 1.00]
   Train: (4000, 784), Test: (1000, 784)

📊 Results (ALL 784 features):
   Accuracy : 58.60%
   Time     : 0.3538s

  BASELINE: MADELON

📦 Loading dataset: madelon ...
   Shape     : (2600, 500)
   Classes   : [0 1]
   Features  : 500
   ✅ Preprocessing done. Feature range: [0.00, 1.00]
   Train: (2080, 500), Test: (520, 500)

🔍 Sanity Check:
   Train Accuracy : 86.54%
   Test  Accuracy : 84.04%

📊 Results (ALL 500 features):
   Accuracy : 84.04%
   Time     : 0.0504s

  BASELINE: CIFAR10_GRAY

📦 Loading dataset: cifar10_gray ...
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 137s 1us/step
   Shape     : (60000, 1024)
   Classes   : [0 1 2 3 4 5 6 7 8 9]
   Features  : 1024
   Subsampled to 5000 samples
   ✅ Preprocessing done. Feature range: [0.00, 1.00]
   Train: (4000, 1024), 

Summary Table

In [6]:
summary = pd.DataFrame([{
    "Dataset":           r["dataset"],
    "Num Features":      r["n_features"],
    "Full Accuracy (%)": round(r["accuracy_full"] * 100, 2),
    "Time (s)":          round(r["time_full"], 4)
} for r in results.values()])

print("\n📋 BASELINE SUMMARY TABLE")
print(summary.to_string(index=False))

summary.to_csv("baseline_results.csv", index=False)
print("\n✅ Saved to baseline_results.csv")


📋 BASELINE SUMMARY TABLE
     Dataset  Num Features  Full Accuracy (%)  Time (s)
       mnist           784              58.60    0.3538
     madelon           500              84.04    0.0504
cifar10_gray          1024              22.70    0.3338

✅ Saved to baseline_results.csv


In [7]:
# ─────────────────────────────────────────────────────────────
# 1. BINARY MASK ENCODING
# ─────────────────────────────────────────────────────────────

class BinaryMaskEncoder:
    """
    Represents feature subsets as binary masks.
    A mask is an array like [1, 0, 1, 1, 0, ...] where:
      1 = keep this feature
      0 = drop this feature
    """

    def __init__(self, n_features: int):
        """
        n_features: total number of features in the dataset
                    e.g. 784 for MNIST, 500 for Madelon, 1024 for CIFAR-10
        """
        self.n_features = n_features

    def all_ones(self):
        """Returns a mask with ALL features selected (baseline)."""
        return np.ones(self.n_features, dtype=int)

    def all_zeros(self):
        """Returns a mask with NO features selected."""
        return np.zeros(self.n_features, dtype=int)

    def random_mask(self, density=0.5, seed=None):
        """
        Returns a random mask.
        density: probability that each feature is set to 1
                 e.g. 0.5 means roughly half the features are selected
        """
        if seed is not None:
            np.random.seed(seed)
        return (np.random.rand(self.n_features) < density).astype(int)

    def flip_bit(self, mask, index):
        """
        Flips a single bit in the mask at position 'index'.
        If it was 1, it becomes 0. If it was 0, it becomes 1.
        Returns a NEW mask (does not change the original).
        """
        new_mask = mask.copy()
        new_mask[index] = 1 - new_mask[index]  # flip: 0→1 or 1→0
        return new_mask

    def count_active(self, mask):
        """Returns how many features are currently selected (how many 1s)."""
        return int(np.sum(mask))

    def get_active_indices(self, mask):
        """Returns the positions of all selected features (where mask == 1)."""
        return np.where(np.array(mask) == 1)[0]

    def mask_to_string(self, mask):
        """Converts mask to a compact string like '101101...' for logging."""
        return ''.join(map(str, mask))

In [8]:
# ─────────────────────────────────────────────────────────────
# 2. FITNESS FUNCTION
# ─────────────────────────────────────────────────────────────

class FitnessFunction:
    """
    Scores a binary mask based on:
      - Accuracy from NaiveBayes (higher = better)
      - Number of features used (fewer = better)

    Final score = accuracy - (alpha * feature_ratio)
    where feature_ratio = selected_features / total_features

    alpha controls how much we penalize using many features.
    A higher alpha = we care more about reducing features.
    """

    def __init__(self, evaluator: NaiveBayesEvaluator, alpha=0.01):
        """
        evaluator : an instance of NaiveBayesEvaluator (Aqib's class)
        alpha     : penalty weight for number of features used
                    default 0.01 means slight preference for fewer features
        """
        self.evaluator = evaluator
        self.alpha = alpha

    def evaluate(self, mask, X_train, X_test, y_train, y_test):
        """
        Given a binary mask, calculate its fitness score.

        Returns:
          score       : fitness value (higher is better)
          accuracy    : raw classification accuracy
          n_selected  : number of features used
          time_taken  : how long evaluation took
        """
        mask = np.array(mask)
        n_total    = len(mask)
        n_selected = int(np.sum(mask))

        # If no features are selected, return worst possible score
        if n_selected == 0:
            return 0.0, 0.0, 0, 0.0

        # Get accuracy from evaluator
        accuracy, time_taken = self.evaluator.evaluate(
            X_train, X_test, y_train, y_test,
            feature_mask=mask
        )

        # feature_ratio: what fraction of features are we using?
        # e.g. 100 out of 784 = 0.127
        feature_ratio = n_selected / n_total

        # Fitness score: reward accuracy, penalize using too many features
        score = accuracy - (self.alpha * feature_ratio)

        return score, accuracy, n_selected, time_taken

In [9]:
# ─────────────────────────────────────────────────────────────
# 3. RESULTS LOGGER
# ─────────────────────────────────────────────────────────────

class ResultsLogger:
    """
    Tracks and saves the performance of every mask that gets evaluated.
    Hasan will use this data to compare against his sequential search.
    """

    def __init__(self, dataset_name: str):
        self.dataset_name = dataset_name
        self.records = []  # list of dicts, one per evaluation

    def log(self, mask, score, accuracy, n_selected, time_taken, label=""):
        """
        Log one evaluation result.

        mask        : the binary mask that was tested
        score       : fitness score
        accuracy    : classification accuracy (0 to 1)
        n_selected  : number of features selected
        time_taken  : evaluation time in seconds
        label       : optional tag like 'random', 'all_features', etc.
        """
        encoder = BinaryMaskEncoder(len(mask))
        self.records.append({
            "dataset"    : self.dataset_name,
            "label"      : label,
            "score"      : round(score, 6),
            "accuracy"   : round(accuracy, 6),
            "n_selected" : n_selected,
            "n_total"    : len(mask),
            "feature_%"  : round(100 * n_selected / len(mask), 2),
            "time_s"     : round(time_taken, 6),
            "mask_str"   : encoder.mask_to_string(mask)
        })

    def get_best(self):
        """Returns the single best result logged so far (highest score)."""
        if not self.records:
            return None
        return max(self.records, key=lambda r: r["score"])

    def to_dataframe(self):
        """Returns all logged results as a pandas DataFrame."""
        return pd.DataFrame(self.records)

    def save_csv(self, filename=None):
        """Saves all results to a CSV file."""
        if filename is None:
            filename = f"fitness_results_{self.dataset_name}.csv"
        df = self.to_dataframe()
        df.to_csv(filename, index=False)
        print(f"✅ Saved {len(df)} results to '{filename}'")
        return filename

    def print_summary(self):
        """Prints a clean summary of all evaluations."""
        df = self.to_dataframe()
        if df.empty:
            print("No results logged yet.")
            return

        print(f"\n📋 RESULTS SUMMARY — {self.dataset_name.upper()}")
        print(f"   Total evaluations : {len(df)}")
        print(f"   Best accuracy     : {df['accuracy'].max()*100:.2f}%")
        print(f"   Best score        : {df['score'].max():.6f}")
        print(f"   Avg features used : {df['n_selected'].mean():.1f} / {df['n_total'].iloc[0]}")
        print(f"\n   Top 5 results:")
        top5 = df.nlargest(5, 'score')[['label','accuracy','n_selected','feature_%','score','time_s']]
        print(top5.to_string(index=False))

In [10]:
# ─────────────────────────────────────────────────────────────
# 4. DEMO — Run on all 3 datasets
# ─────────────────────────────────────────────────────────────

def run_fitness_demo(dataset_name, result_dict, n_random_masks=10, seed=42):
    """
    Demonstrates the fitness function on a dataset by testing:
      - All features (baseline mask)
      - Several random masks
    """
    print(f"\n{'='*55}")
    print(f"  FITNESS DEMO: {dataset_name.upper()}")
    print(f"{'='*55}")

    # Unpack data
    X_train = result_dict["X_train"]
    X_test  = result_dict["X_test"]
    y_train = result_dict["y_train"]
    y_test  = result_dict["y_test"]
    n_features = X_train.shape[1]

    # Set up three components
    encoder  = BinaryMaskEncoder(n_features)
    evaluator = NaiveBayesEvaluator()
    fitness  = FitnessFunction(evaluator, alpha=0.01)
    logger   = ResultsLogger(dataset_name)

    # Test 1: All features
    mask_all = encoder.all_ones()
    score, acc, n_sel, t = fitness.evaluate(mask_all, X_train, X_test, y_train, y_test)
    logger.log(mask_all, score, acc, n_sel, t, label="all_features")
    print(f"\n✅ All features   → Accuracy: {acc*100:.2f}%  Score: {score:.4f}  Features: {n_sel}/{n_features}")

    # Test 2: random masks 
    np.random.seed(seed)
    print(f"\n Testing {n_random_masks} random masks (50% density)...")
    for i in range(n_random_masks):
        mask = encoder.random_mask(density=0.5)
        score, acc, n_sel, t = fitness.evaluate(mask, X_train, X_test, y_train, y_test)
        logger.log(mask, score, acc, n_sel, t, label=f"random_{i+1}")
        print(f"   Mask {i+1:2d}: Accuracy={acc*100:.2f}%  Features={n_sel}/{n_features}  Score={score:.4f}")

    # Print summary and save
    logger.print_summary()
    logger.save_csv()

    return logger


# RUN ON ALL 3 DATASETS

fitness_loggers = {}
for ds_name in ["mnist", "madelon", "cifar10_gray"]:
    fitness_loggers[ds_name] = run_fitness_demo(
        dataset_name  = ds_name,
        result_dict   = results[ds_name],
        n_random_masks= 10
    )

print("\n\n🎉 All CSV files saved")


  FITNESS DEMO: MNIST

✅ All features   → Accuracy: 58.60%  Score: 0.5760  Features: 784/784

 Testing 10 random masks (50% density)...
   Mask  1: Accuracy=59.20%  Features=387/784  Score=0.5871
   Mask  2: Accuracy=52.70%  Features=388/784  Score=0.5221
   Mask  3: Accuracy=59.20%  Features=385/784  Score=0.5871
   Mask  4: Accuracy=58.50%  Features=395/784  Score=0.5800
   Mask  5: Accuracy=53.80%  Features=395/784  Score=0.5330
   Mask  6: Accuracy=57.80%  Features=389/784  Score=0.5730
   Mask  7: Accuracy=55.80%  Features=416/784  Score=0.5527
   Mask  8: Accuracy=55.30%  Features=394/784  Score=0.5480
   Mask  9: Accuracy=54.30%  Features=404/784  Score=0.5378
   Mask 10: Accuracy=55.90%  Features=403/784  Score=0.5539

📋 RESULTS SUMMARY — MNIST
   Total evaluations : 11
   Best accuracy     : 59.20%
   Best score        : 0.587089
   Avg features used : 430.9 / 784

   Top 5 results:
       label  accuracy  n_selected  feature_%    score   time_s
    random_3     0.592        